In [ ]:
# %%
import pickle
import pandas as pd
from IPython.display import display, Markdown

# ==========================================
# ⏱️ 量化时光机：机构级每日战报
# ==========================================
TARGET_DATE = "2026-04-28"
report_file = f"Daily_Reports/Report_{TARGET_DATE}.pkl"

try:
    with open(report_file, 'rb') as f:
        snap = pickle.load(f)

    # 标题头
    display(Markdown(f"# 📊 宽客实战日报 | 交易日: {snap['date']}"))
    display(
        Markdown(f"### 👑 今日市场绝对主线 (King Factor): **`{snap['king_factor']}`**"))
    display(Markdown("---"))

    # 模块 1：OLS 市场风向归因
    r2_fac = snap['ols_report']['r2_factors'] * 100
    r2_ind = snap['ols_report']['r2_ind'] * 100
    display(Markdown(
        f"#### 📡 1. 市场微观结构探测 (解释力 - 因子: `{r2_fac:.2f}%` | 行业: `{r2_ind:.2f}%`)"))

    # ✨ 修复了 Pandas 的版本兼容警告：用 map 替代了 applymap
    style_ols = snap['ols_report']['stats_df'].style.map(
        lambda x: 'color: red; font-weight: bold' if isinstance(x, str) and '追捧 ★' in x else
                  ('color: green; font-weight: bold' if isinstance(x, str)
                   and '抛售 ★' in x else ''),
        subset=['风向状态']
    ).background_gradient(subset=['Beta系数'], cmap='coolwarm', vmin=-0.05, vmax=0.05)
    display(style_ols)

    # 模块 1.5：因子强度“英雄榜”
    display(
        Markdown(f"#### 🏆 1.5 因子全维度强度对比 (当前选股基准: `{snap['king_factor']}`)"))

    # 对统计表进行美化：Beta 用渐变色，T值绝对值大于 1.96 的加粗
    king_stats_styled = snap['king_stats'].style.background_gradient(
        subset=['Beta (强度)'], cmap='RdYlGn'
    ).map(
        lambda x: 'font-weight: bold; color: #FF4500' if isinstance(
            x, (int, float)) and abs(x) > 1.96 else '',
        subset=['T值 (显著性)']
    )

    display(king_stats_styled)

    # 计算次强因子与强度差距
    stats_df = snap['king_stats']
    if len(stats_df) > 1:
        top_1 = stats_df.iloc[0]
        top_2 = stats_df.iloc[1]
        gap = round(top_1['Beta (强度)'] - top_2['Beta (强度)'], 4)
        display(Markdown(
            f"> 💡 **复盘笔记**：今日最强因子 `{top_1['因子名称']}` 与次强因子 `{top_2['因子名称']}` 的强度差距为 **`{gap}`**。"))

    # 模块 2：强势板块与龙头画像
    display(Markdown("#### 🎯 2. 强势板块下钻与龙头画像"))
    # 横向展示领涨和领跌板块
    col1, col2 = snap['drill_report']['top_industries'], snap['drill_report']['bottom_industries']
    display(pd.concat([col1.set_index('板块名称'), col2.set_index(
        '板块名称')], axis=1, keys=['🔥 领涨板块', '❄️ 领跌板块']))

    display(Markdown("**【强势板块内部资金画像】**"))
    display(snap['drill_report']['portraits_df'].style.set_properties(
        **{'text-align': 'left'}))
    display(Markdown("---"))

    # 模块 3：明日交易底仓狙击池
    display(
        Markdown(f"#### 🔫 3. 明日开盘狙击池 (按核心因子 **`{snap['king_factor']}`** 降序排列)"))

    display_cols = ['code', 'name', snap['king_factor'],
                    'Momentum', 'Liquidity', 'Size']
    # 将更多你想看的因子加到表格里显示 (比如把 Amihud 也展示出来)：
    # display_cols = ['code', 'name', snap['king_factor'], 'Momentum', 'Liquidity', 'Size', 'Amihud']

    display_cols = list(dict.fromkeys(display_cols))

    # 原来是 .head(10)，改成 .head(20) 就能看前 20 名金股！
    top_picks = snap['top_picks'].head(20)[display_cols]

    display(top_picks.style.background_gradient(
        subset=[snap['king_factor']], cmap='YlOrRd'))


except FileNotFoundError:
    print(f"❌ 找不到 {TARGET_DATE} 的战报！")
# %%

# 📊 宽客实战日报 | 交易日: 2026-04-27

### 👑 今日市场绝对主线 (King Factor): **`Momentum`**

---

#### 📡 1. 市场微观结构探测 (解释力 - 因子: `3.93%` | 行业: `18.60%`)

,因子 (Factor),Beta系数,T值 (显著性),风向状态
0,Momentum,20.066000,3.330000,追捧 ★
1,Short_Rev,19.184000,3.080000,追捧 ★
2,Low_Vol,-12.045000,-1.920000,抛售
3,Liquidity,-0.948000,-0.160000,抛售
4,Size,7.887000,1.250000,追捧
5,Value_BP,5.375000,0.630000,追捧
6,Mom_Sharpe,16.219000,2.460000,追捧 ★
7,Vol_Price_Corr,-2.161000,-0.360000,抛售
8,Amihud,-3.043000,-0.470000,抛售


...

**【强势板块内部资金画像】**

,强势板块,代码,名称,预期超额涨幅,核心因子画像
0,B06煤炭开采和洗选业,sh.601918,新集能源,+12.99%,中庸(随板块普涨)
1,B06煤炭开采和洗选业,sh.601699,潞安环能,+11.99%,"低Momentum(-1.2), 高Value_BP(+1.5), 低Mom_Sharpe(-1.2), 低Vol_Price_Corr(-1.2)"
2,B06煤炭开采和洗选业,sh.600188,兖矿能源,+10.87%,"低Momentum(-1.4), 高Short_Rev(+1.6), 高Size(+1.1), 低Mom_Sharpe(-1.6), 低Vol_Price_Corr(-1.4)"
3,B06煤炭开采和洗选业,sh.600985,淮北矿业,+8.18%,高Value_BP(+1.5)
4,B06煤炭开采和洗选业,sh.601898,中煤能源,+8.03%,"低Momentum(-1.6), 高Short_Rev(+1.9), 高Size(+1.4), 低Mom_Sharpe(-1.7), 低Vol_Price_Corr(-1.5)"
5,B06煤炭开采和洗选业,sh.600546,山煤国际,+7.78%,中庸(随板块普涨)
6,B06煤炭开采和洗选业,sh.601666,平煤股份,+7.24%,"高Value_BP(+1.6), 低Mom_Sharpe(-1.0)"
7,B06煤炭开采和洗选业,sh.601001,晋控煤业,+6.15%,"低Mom_Sharpe(-1.1), 低Vol_Price_Corr(-2.1)"
8,B06煤炭开采和洗选业,sz.000983,山西焦煤,+4.11%,"低Momentum(-1.2), 高Short_Rev(+1.0), 高Value_BP(+1.2), 低Mom_Sharpe(-1.2)"
9,B06煤炭开采和洗选业,sz.000937,冀中能源,+3.66%,"高Low_Vol(+1.5), 高Value_BP(+1.1), 低Mom_Sharpe(-1.5)"


---

#### 🔫 3. 明日开盘狙击池 (按核心因子 **`Momentum`** 降序排列)

,code,name,Momentum,Liquidity,Size
0,300308,中际旭创,3.253430,0.462602,3.111902
1,688002,睿创微纳,2.758893,0.015893,0.194694
2,601138,工业富联,2.709134,-0.509817,3.111902
3,688041,海光信息,2.566199,-0.395485,2.881689
4,688256,寒武纪,2.430904,0.111749,2.723779
5,688008,澜起科技,2.374030,1.242999,1.486656
6,977,浪潮信息,2.006157,1.946125,0.857907
7,688347,华虹公司,1.973935,1.412965,0.078226
8,2466,天齐锂业,1.971569,1.893386,0.803192
9,338,潍柴动力,1.898911,0.322086,1.187007
